# Evaluation Report for the paper "_dlinear_: Enhancing SMT Solvers with Floating-Point Exact LP Solvers"

---

## 1. Executive Summary

This report presents a comprehensive comparison between the baseline [cvc5](https://github.com/cvc5/cvc5) SMT solver, the existing implementation that uses [GLPK](https://www.gnu.org/software/glpk/) as an external floating-point LP solver, and our proposed approach, _dlinear_, which uses floating-point exact LP solvers, specifically [SoPlex](https://github.com/scipopt/soplex) and [qsoptex](https://github.com/TendTo/qsopt-ex).
Additionally, we compare against the state-of-the-art SMT solvers [Z3](https://github.com/Z3Prover/z3) and [yices](https://github.com/SRI-CSL/yices) to contextualize the performance of our approach in the broader SMT landscape.

The evaluation demonstrates that our integration provides significant performance improvements, particularly for instances that require intensive linear arithmetic reasoning.

For more details on the contributions, please refer to the paper.

---

In [ ]:
from IPython.display import Markdown as md
import pandas as pd
from functools import reduce
from utils import (
    compare_unique_solved_instances,
    build_markdown_comparison_matrix,
    plot_performance_profiles,
    sanitize,
    difficulty_analysis,
    print_stats,
    SolverResult,
    external_solver_impact,
    FlowLayout,
)
from itertools import product
import os
import matplotlib.pyplot as plt

DROP_UNSOLVED = False
SOLVERS = ("soplex", "qsoptex")
ITERATIONS = (100, 200,  300)
MODES = ("", "_strict",)
SOLVERS = ("soplex", "qsoptex")
ITERATIONS_DELTA = [0]
MODES_DELTA = ("", "_delta1e+30")

def get_tot_results_compare(result: SolverResult, iterations_filter: list[tuple[int, pd.DataFrame]]) -> int:
    header = ""
    middle = ""
    row = ""
    for iterations, instances in iterations_filter:
        results = result.dataframe[result.dataframe.index.isin(instances.index)]
        results = results[results[result.result_key].isin(["sat", "unsat"])]
        not_solved = len(instances) - len(results)
        header += f"| $n={iterations}$ solved | $n={iterations}$ unsolved "
        middle += f"| --- | --- "
        row += f"| {len(results)}/ {len(instances)} ({(len(results) / len(instances) * 100):.1f}%) | {not_solved}/ {len(instances)} ({(not_solved / len(instances) * 100):.1f}%) "
    return f"""
#### {result.solver_name} results for all pivots

| Solver               {header} |
| -------------------- {middle} |
| {result.solver_name} {row} |
"""

def difficulty_analysis_by_group(group: pd.DataFrame, iterations: int, group_name: str = "QF_LRA"):
    if iterations == 100:
        instances_filter = instances_100
        return md(difficulty_analysis(
            solvers_analysis=[
                others_results.get("yices", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("z3", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("cvc5", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("glpk_i100", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("soplex_i100", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("soplex_i100_strict", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("qsoptex_i100", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("qsoptex_i100_strict", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
             ],
            total_count=len(pd.merge(group, instances_filter, on="file", how="inner")),
            group_name=f"{group_name} with $n = 100$ pivots threshold"
        ))
    elif iterations == 200:
        instances_filter = instances_200
        return md(difficulty_analysis(
            solvers_analysis=[
                others_results.get("yices", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("z3", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("cvc5", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("glpk_i200", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("soplex_i200", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("soplex_i200_strict", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("qsoptex_i200", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("qsoptex_i200_strict", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
             ],
            total_count=len(pd.merge(group, instances_filter, on="file", how="inner")),
            group_name=f"{group_name} with $n = 200$ pivots threshold"
        ))
    elif iterations == 300:
        instances_filter = instances_300
        return md(difficulty_analysis(
            solvers_analysis=[
                others_results.get("yices", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("z3", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("cvc5", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                others_results.get("glpk_i300", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("soplex_i300", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("soplex_i300_strict", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("qsoptex_i300", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
                results.get("qsoptex_i300_strict", SolverResult.empty()).apply_filter(lambda df: in_instances(df, group, instances_filter)),
             ],
            total_count=len(pd.merge(group, instances_filter, on="file", how="inner")),
            group_name=f"{group_name} with $n = 300$ pivots threshold"
        ))

# Load the data
all_instances = pd.read_csv("instances/all_instances.csv").set_index("file")
instances_100 = pd.read_csv("instances/100_instances.csv").set_index("file")
instances_200 = pd.read_csv("instances/200_instances.csv").set_index("file")
instances_300 = pd.read_csv("instances/300_instances.csv").set_index("file")
miplib = pd.read_csv("instances/miplib.csv").set_index("file")
latendresse = pd.read_csv("instances/latendresse.csv").set_index("file")
dtp_scheduling = pd.read_csv("instances/dtp-scheduling.csv").set_index("file")
sk = pd.read_csv("instances/sk_instances.csv").set_index("file")

instances_dict = {
    100: instances_100,
    200: instances_200,
    300: instances_300,
    "sk": sk,
    0: sk,
}

def in_n(df: pd.DataFrame, n: int) -> pd.DataFrame:
    return df[df.index.isin(instances_dict[n].index)]

def in_instances(df: pd.DataFrame, *instances: pd.DataFrame) -> pd.DataFrame:
    if len(instances) == 0:
        return df
    if len(instances) == 1:
        return df[df.index.isin(instances[0].index)]
    return df[df.index.isin(pd.merge(*instances, how="inner", left_index=True, right_index=True).index)]

results: dict[str, SolverResult] = {}
for solver, iteration, m in product(SOLVERS, ITERATIONS, MODES):
    key = f"{solver}_i{iteration}{m}"
    file = f"results/{key}.csv"
    solver_id = solver[0].upper()
    if os.path.exists(file):
        df = pd.read_csv(file).set_index("file")
        df = df[df["theory::arith::z::arith::relax::calls"].astype(int) > 0]
        if DROP_UNSOLVED:
            df = df[df[f"result{solver_id}"].isin(["sat", "unsat"])]
        if m == "_strict":
            mode_str = "t"
        elif m == "_delta1e-08":
            mode_str = r"\delta"
        else:
            mode_str = r"\varepsilon"
        results[key] = SolverResult(solver_name=fr"$ \textbf{{d}}_{{{solver[:2].lower()},{iteration}}}^{mode_str} $", dataframe=df, solver_id=solver_id, iterations=iteration)

for key in results:
    results[key] = results[key].replace_df(sanitize(results[key].dataframe))

results_delta: dict[str, SolverResult] = {}
for solver, iteration, m in product(SOLVERS, ITERATIONS_DELTA, MODES_DELTA):
    key = f"{solver}_i{iteration}{m}"
    file = f"results/delta/{key}.csv"
    solver_id = solver[0].upper()
    if os.path.exists(file):
        df = pd.read_csv(file).set_index("file")
        df = df[df["theory::arith::z::arith::relax::calls"].astype(int) > 0]
        df = df[df.index.isin(instances_dict["sk"].index)]
        df = df[((df["theory::arith::z::approx::delta"] > 0) & (df["theory::arith::z::approx::delta"] > 0)) | (df["theory::arith::z::approx::delta"] == -1)]
        if DROP_UNSOLVED:
            df = df[df[f"result{solver_id}"].isin(["sat", "unsat"])]
        if m == "_strict":
            mode_str = "t"
        elif "_delta" in m:
            mode_str = r"{\delta_{" + m.split("_delta")[1] + "}}"
            mode_str = r"{\delta}"
        else:
            mode_str = r"\varepsilon"
        results_delta[key] = SolverResult(solver_name=fr"$ \textbf{{d}}_{{{solver[:2].lower()},{iteration}}}^{mode_str} $", dataframe=df, solver_id=solver_id, iterations=iteration)

for key in results_delta:
    results_delta[key] = results_delta[key].replace_df(sanitize(results_delta[key].dataframe))

others = ("delta/glpk_i0", "glpk_i100", "glpk_i200", "glpk_i300", "yices", "z3", "cvc5", )
others_results: dict[str, SolverResult] = {}
for others in others:
    if os.path.exists(f"results/{others}.csv"):
        df = pd.read_csv(f"results/{others}.csv").set_index("file")
        if "/" in others:
            others = others.split("/")[-1]
        if DROP_UNSOLVED:
            df = df[df[f"result{others[0].upper()}"].isin(["sat", "unsat"])]
        iterations = f"_{{{others.split('_')[-1][1:]}}}" if "glpk" in others else ""
        df = df[df["theory::arith::z::arith::relax::calls"].astype(int) > 0] if "glpk" in others else df
        others_results[others] = SolverResult(solver_name=fr"$ \textbf{{{others[0].lower()}}}{iterations} $", dataframe=df, solver_id=others[0].upper(), iterations=int(iterations[2:-1]) if iterations else -1)

FlowLayout.init_css()

## 2. Overall Benchmark Performance

### 2.1 Problems Solved

This section summarizes the total number of instances solved by each approach across the entire QF_LRA benchmark set.
We use the [SMT-LIB](https://smt-lib.org/benchmarks.shtml) benchmarks for evaluation, restricting ourselves to the [QF_LRA](https://smt-lib.org/logics-all.shtml#QF_LRA) logic from the [SMT-LIB release 2025 of non-incremental benchmarks](https://zenodo.org/records/16740866).
The complete benchmark set consists of 1,753 instances. However, our evaluation uses only a subset, namely those instances that invoke the external LP solver at least once.

### 2.2 Experimental Setup

- **Timeout**: 6 hours per instance
- **Memory limit**: 4 GB
- **Hardware**: Comet HPC cluster (Newcastle University), using a single core of an AMD EPYC 9745 processor @ 2.4 GHz

### 2.3 Hardware Details

| OS            | CPU           | Architecture | Frequency | Turbo Frequency | Cores | Memory | L1      | L2        | L3     | Time Limit |
| ------------- | ------------- | ------------ | --------- | --------------- | ----- | ------ | ------- | --------- | ------ | ---------- |
| RHEL 11.4.1-4 | AMD EPYC 9745 | x86_64       | 2.4 GHz   | 3.7 GHz         | 1     | 4 GB   | 6144 KB | 131072 KB | 256 MB | 6 hours    |

In [ ]:
header = "| Total number of instances "
middle = "|:---:"
key = f"| {len(all_instances)} "
for iterations in ITERATIONS:
    header += f"| Instances with at least one external LP solver call with $n={iterations}$ iterations threshold "
    middle += "|:---:"
    key += f"| {len(in_n(all_instances, iterations))} "
header += "|"
middle += "|"
key += "|"

md(f"""### Instances
   
Starting from the original {len(all_instances)} instances, the table below shows how many instances required at least a call to the external LP solver for the given iteration threshold.
   
{header}
{middle}
{key}
   """)

### 2.4 Solver Notation

When abbreviating the solvers, we use the following conventions:

- $\textbf{c}$, $\textbf{y}$, and $\textbf{z}$ for _cvc5_, _yices_, and _z3_, respectively
- $\textbf{g}_n$ for _glpk_, where $n \in \{0, 100, 200, 300\}$ is the number of pivots performed by the rational simplex implementation before calling the external LP solver
- $\textbf{d}^m_{s,n}$ for _dlinear_ in complete mode, where:
  - $s \in \{\text{so}, \text{qs}\}$ indicates the external LP solver (_soplex_ or _qsoptex_)
  - $m \in \{\varepsilon, t\}$ indicates how strict inequalities are handled ($\varepsilon$ perturbation on strict constraints or use of a strict variable $t$)
  - $n$ is the pivot threshold as described above
- $\textbf{d}^{\delta}_{s, n}$ for _dlinear_ in $\delta$-complete mode, where $s$ and $n$ are as described above

See _Section 6_ of the paper for more details.

In [ ]:
md(
    fr"""### 2.5 Loading _dlinear_ results
   
In the following table, we summarize the loaded results for each solver configuration. 
Each row corresponds to a specific configuration of our _dlinear_ approach, with different pivot thresholds (100, 200, 300) and modes (epsilon, strict, delta). 
The "Loaded Instances" column indicates how many benchmark instances were successfully loaded for each configuration.
Higher pivot thresholds are expected to have less loaded instances, as we only consider those that performed at least one call to the external LP solver.

The _Mode_ column indicates how constraints with strict inequalities (i.e., $<, >$) are handled.
The possible modes are:
   
- $\varepsilon$: The default mode, which uses a small perturbation $\varepsilon$ on the right-hand side of the constraints to ensure strict inequalities are satisfied.
- $t$: A mode that adds and additional strict variable $t$ to constraint containing strict inequalities, and then tries to maximize $t$ to ensure that the constraints are satisfied with a positive margin.
- $\delta$: $\delta$-complete mode, which uses a user-defined $\delta$ value to relax the constraints, effectively transforming strict inequalities into non-strict ones.
   
Additional details can be found in _Sections 4.2_ of the paper.
   
| Solver | Mode |  Pivot Threshold | Available Instances | Solved Instances | Unsolved Instances |
|--------|-----------------|------|-------------------| -----------------|-------------------|
{"\n".join(f"| {res.solver_name} | {res.mode}  | {res.iterations} | {len(in_n(all_instances, res.iterations))} | {len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])} ({len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])]) / len(in_n(all_instances, res.iterations)) * 100:.1f}%) | {len(in_n(all_instances, res.iterations)) - len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])} ({(len(in_n(all_instances, res.iterations)) - len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])) / len(in_n(all_instances, res.iterations)) * 100:.1f}%) |" for res in results.values())}
{"\n".join(f"| {res.solver_name} | {res.mode}  | {res.iterations} | {len(sk)} | {len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])} ({len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])]) / len(sk) * 100:.1f}%) | {len(sk) - len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])} ({(len(sk) - len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])) / len(sk) * 100:.1f}%) |" for res in results_delta.values() if not res.dataframe.empty)}
"""
)

### 2.6 Comparison between external LP solvers: GLPK, SoPlex, and qsoptex

We compare the performance of these three external LP solvers when integrated into cvc5, across different pivot thresholds and operational modes.

In [ ]:
md(print_stats(
    soplex_configs=(list(results.values()) + [val for val in others_results.values() if val.solver_id == "G"] + list(results_delta.values())),
))

_Extended **Table 1** from the paper._

### 2.7 External Solver Impact

Calls to the external simplex solver can significantly affect the overall performance of the SMT solver, particularly when precision boosting and iterative refinement techniques are costly, which is common for numerically challenging instances. Here, we analyze the impact of:

- **Precision boosting**: The number $p_n$ of calls that required $n$ bits of precision
- **Iterative refinements**: The number $r_i$ of calls that required $i$ refinement iterations

This analysis helps quantify the overhead introduced by exact arithmetic techniques.

In [ ]:
md(external_solver_impact(
    solvers_analysis=list(results.values()) + list(results_delta.values()),
    fig_caption=r"$\textit{Figure summary of the combined \textbf{Table 2} and \textbf{Table 7} from the paper}$"
))

_Combined **Table 2** and **Table 7** from the paper._

## 3. Difficulty and Performance Analysis

### 3.1 Difficulty Analysis

We analyze the distribution of solving times across benchmark instances, dividing them into different buckets based on the time taken to solve each instance.

In [ ]:
difficulty_analysis(
    solvers_analysis=[
        others_results.get("yices", SolverResult.empty()).apply_filter(lambda df: in_n(df, 100)),
        others_results.get("z3", SolverResult.empty()).apply_filter(lambda df: in_n(df, 100)),
        others_results.get("cvc5", SolverResult.empty()).apply_filter(lambda df: in_n(df, 100)),
        others_results.get("glpk_i100", SolverResult.empty()),
        results.get("soplex_i100", SolverResult.empty()),
        results.get("soplex_i100_strict", SolverResult.empty()),
        # results.get("soplex_i100_delta1e-08"),
        results.get("qsoptex_i100", SolverResult.empty()),
        results.get("qsoptex_i100_strict", SolverResult.empty()),
        # results.get("qsoptex_i100_delta1e-08"),
    ],
    total_count= len(instances_100),
    group_name="instances with $n = 100$ pivots threshold",
    fig_caption=r"$\textit{\textbf{Figure 2 (a)} from the paper}$"
)

In [ ]:
difficulty_analysis(
    solvers_analysis=[
        others_results.get("yices",SolverResult.empty()).apply_filter(lambda df: in_n(df, 200)),
        others_results.get("z3",SolverResult.empty()).apply_filter(lambda df: in_n(df, 200)),
        others_results.get("cvc5",SolverResult.empty()).apply_filter(lambda df: in_n(df, 200)),
        others_results.get("glpk_i200",SolverResult.empty()),
        results.get("soplex_i200",SolverResult.empty()),
        results.get("soplex_i200_strict",SolverResult.empty()),
        # results.get("soplex_i200_delta1e-08"),
        results.get("qsoptex_i200",SolverResult.empty()),
        results.get("qsoptex_i200_strict",SolverResult.empty()),
        # results.get("qsoptex_i200_delta1e-08"),
    ],
   total_count= len(instances_200),
   group_name="instances with $n = 200$ pivots threshold",
   fig_caption=r"$\textit{\textbf{Figure 2 (b)} from the paper}$"
)

In [ ]:
difficulty_analysis(
    solvers_analysis=[
        others_results.get("yices", SolverResult.empty()).apply_filter(lambda df: in_n(df, 300)),
        others_results.get("z3", SolverResult.empty()).apply_filter(lambda df: in_n(df, 300)),
        others_results.get("cvc5", SolverResult.empty()).apply_filter(lambda df: in_n(df, 300)),
        others_results.get("glpk_i300", SolverResult.empty()),
        results.get("soplex_i300", SolverResult.empty()),
        results.get("soplex_i300_strict", SolverResult.empty()),
        results.get("qsoptex_i300", SolverResult.empty()),
        results.get("qsoptex_i300_strict", SolverResult.empty()),
    ],
   total_count= len(instances_300),
   group_name="instances with $n = 300$ pivots threshold",
   fig_caption=r"$\textit{Figure summary of \textbf{Table 3 (below)} from the paper}$"
)

### 3.2 Performance Profiles

Performance profiles are a powerful tool for comparing the performance of multiple solvers across a benchmark set.
They highlight the percentage of instances each solver can complete within a given time factor of the best solver for each instance, providing a clear visualization of relative performance.

In [ ]:
def to_new_solver_id(solver_result: SolverResult):
    return f"{solver_result.solver_id}{solver_result.iterations}{'S' if '^t' in solver_result.solver_name.lower() else ''}{'D' if r'\delta' in solver_result.solver_name.lower() else ''}"

cvc5_variants_results = [
    SolverResult(
        dataframe=solver_result.dataframe.reset_index()[
            ["file", f"time{solver_result.solver_id}", f"result{solver_result.solver_id}", *(["options::pivots"] if "options::pivots" in solver_result.dataframe.columns else [])]
        ]
        .rename(
            columns={
                f"time{solver_result.solver_id}": f"time{to_new_solver_id(solver_result)}",
                f"result{solver_result.solver_id}": f"result{to_new_solver_id(solver_result)}",
            }
        )
        .set_index("file"),
        solver_name=solver_result.solver_name,
        solver_id=to_new_solver_id(solver_result),
        iterations=solver_result.iterations,
    )
    for solver_result in (results | others_results | results_delta).values()
    if solver_result.solver_id in ["C", "S", "Q", "G"]
]

other_profile_results = [
    SolverResult(
        dataframe=solver_result.dataframe.reset_index()[
            ["file", f"time{solver_result.solver_id}", f"result{solver_result.solver_id}"]
        ]
        .rename(
            columns={
                f"time{solver_result.solver_id}": f"time{to_new_solver_id(solver_result)}",
                f"result{solver_result.solver_id}": f"result{to_new_solver_id(solver_result)}",
            }
        )
        .set_index("file"),
        solver_name=solver_result.solver_name,
        solver_id=to_new_solver_id(solver_result),
        iterations=solver_result.iterations,
    )
    for solver_result in others_results.values()
    if solver_result.solver_id in ["Y", "Z"]
]

# Get the set of instances where any of the SoPlex variants makes external simplex calls, and filter all profiles to only those instances
external_cvc5_variants_results: dict[int, list[SolverResult]] = {}
external_profile_results: dict[int, list[SolverResult]] = {}
for iteration in (*ITERATIONS, 0):
    external_cvc5_variants_results[iteration] = [
        result.replace_df(in_n(result.dataframe.drop(columns=["options::pivots"], errors="ignore"), iteration))
        for result in cvc5_variants_results
        if not result.dataframe.empty and ("options::pivots" not in result.dataframe.columns or result.dataframe["options::pivots"].iloc[0] == iteration)
    ]
    external_profile_results[iteration] = external_cvc5_variants_results[iteration] + [
        result.replace_df(in_n(result.dataframe, iteration)) for result in other_profile_results
    ]

oPlot = FlowLayout()  # create an empty FlowLayout
for caption, profile in zip(
    (
        r"$\textit{Detail of \textbf{Figure 3 (a)} from the paper, only considering CVC5 and dlinear}$",
        r"$\textit{\textbf{Figure 3 (a)} from the paper}$",
        r"$\textit{Detail of \textbf{Figure 3 (b)} from the paper, only considering CVC5 and dlinear}$",
        r"$\textit{\textbf{Figure 3 (b)} from the paper}$",
        r"$\textit{Detail of \textbf{Figure 5} from the paper, only considering CVC5 and dlinear}$",
        r"$\textit{\textbf{Figure 5} from the paper}$",
        r"$\textit{\textbf{Figure 6} from the paper}$",
    ),
    (
        external_cvc5_variants_results.get(100, []),
        external_profile_results.get(100, []),
        external_cvc5_variants_results.get(200, []),
        external_profile_results.get(200, []),
        external_cvc5_variants_results.get(300, []),
        external_profile_results.get(300, []),
        external_profile_results.get(0, []),
    ),
):
    if len(profile) < 2:
        fig, ax = plt.subplots()
        ax.text(0.5, 0.5, "No data to display", horizontalalignment="center", verticalalignment="center", transform=ax.transAxes)
        ax.set_axis_off()
    else:
        ax, _ = plot_performance_profiles(
            *profile,
            accepted_results=["sat", "unsat"],
            result_cols="result",
            metric_cols="time",
            max_tau=1000,
            num_points=1000,
            fig_caption=caption,
            margin_top=2,
        )
    oPlot.add_plot_col(ax)
oPlot.to_html()

### 3.3 All results

We present the complete results for all solvers across all instances in the benchmark set, providing a comprehensive view of the performance landscape.
These are the same tables included in the paper as **Table 8**, **Table 9**, and **Table 10**.

In [ ]:
all_results = []
column_map = {
    "timeY-1": r"yices",
    "timeZ-1": r"z3",
    "timeC-1": r"cvc5",
    "timeG0": r"cvc5+glpk (0)",
    "timeG100": r"cvc5+glpk (100)",
    "timeG200": r"cvc5+glpk (200)",
    "timeG300": r"cvc5+glpk (300)",
    "timeS100": r"dlinear+soplex (e) (100)",
    "timeS100S": r"dlinear+soplex (t) (100)",
    "timeS200": r"dlinear+soplex (e) (200)",
    "timeS200S": r"dlinear+soplex (t) (200)",
    "timeS300": r"dlinear+soplex (e) (300)",
    "timeS300S": r"dlinear+soplex (t) (300)",
    "timeQ100": r"dlinear+qsoptex (e) (100)",
    "timeQ100S": r"dlinear+qsoptex (t) (100)",
    "timeQ200": r"dlinear+qsoptex (e) (200)",
    "timeQ200S": r"dlinear+qsoptex (t) (200)",
    "timeQ300": r"dlinear+qsoptex (e) (300)",
    "timeQ300S": r"dlinear+qsoptex (t) (300)",
    "timeS0D": r"dlinear+soplex (d) (sk)",
    "timeS0": r"dlinear+soplex (e) (sk)",
    "timeQ0D": r"dlinear+qsoptex (d) (sk)",
    "timeQ0": r"dlinear+qsoptex (e) (sk)",
}
for iteration in (*ITERATIONS, 0):
    df_all = reduce(
        lambda left, right: pd.merge(left, right, left_index=True, right_index=True, how="outer"), (in_n(result.dataframe, iteration) for result in external_profile_results[iteration])
    )
    df_all_column_map = {col: name for col, name in column_map.items() if col in df_all.columns}
    df_all["problem_index"] = df_all.index
    df_all["problem_index"] = df_all["problem_index"].apply(lambda x: (pd.concat([instances_dict[100].index.to_series(), instances_dict["sk"].index.to_series()]).index.get_loc(x)))
    df_all["Problem"] = df_all.index
    df_all["Problem"] = df_all["Problem"].apply(lambda x: f"({pd.concat([instances_dict[100].index.to_series(), instances_dict["sk"].index.to_series()]).index.get_loc(x)})")
    all_results.append(df_all.sort_values("problem_index")[list(df_all_column_map.keys())].rename(columns=df_all_column_map))
all_results[0]

In [ ]:
all_results[1]

In [ ]:
all_results[2]

In [ ]:
all_results[3]

In [ ]:
all_results[3].sort_values(["dlinear+qsoptex (d) (sk)", "dlinear+qsoptex (e) (sk)", "dlinear+soplex (e) (sk)"], na_position="last").head(20)